# ARC Prize 2026 - ARC-AGI-2 Submission (v13)

**Team**: karales  
**Model**: Qwen3-8B (4-bit BnB on T4 x2)  
**Pipeline**: Transduction-first solver - strict + relaxed only (v13 speed mode)  
**v13 fixes**: OOM prevention, aggressive memory cleanup, disabled D4/augmented voting, reduced timeouts  

v12 results: 11/23 solved in 7.8h, hit OOM wall at task 9, too slow for 240 tasks.  
v13 target: process all 240 tasks within 12h by skipping expensive phases on failures.

In [ ]:
# Cell 1: Environment setup (v13)
import os, sys, warnings, time, json, logging, gc

# v13: Reduce CUDA memory fragmentation
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('arc_submission')

# Detect environment
ON_KAGGLE = os.path.exists('/kaggle/input')
print(f"Environment: {'Kaggle' if ON_KAGGLE else 'Local'}")
print(f"Python: {sys.version}")

# Add solver library to path
solver_found = False
if ON_KAGGLE:
    DATASET_PATH = '/kaggle/input/datasets/karales/loopagi-arc-solver'
    
    if os.path.exists(DATASET_PATH) and os.path.isdir(os.path.join(DATASET_PATH, 'loopagi')):
        sys.path.insert(0, DATASET_PATH)
        print(f'Solver: added {DATASET_PATH} to sys.path')
        print(f'  Contents: {os.listdir(DATASET_PATH)}')
        print(f'  loopagi/: {os.listdir(os.path.join(DATASET_PATH, "loopagi"))[:10]}')
        solver_found = True
    else:
        # Fallback: search for loopagi package anywhere under /kaggle/input
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'loopagi' in dirs and os.path.isdir(os.path.join(root, 'loopagi', 'arc')):
                sys.path.insert(0, root)
                print(f'Solver: added {root} to sys.path (fallback search)')
                solver_found = True
                break
    
    if not solver_found:
        print('WARNING: loopagi solver package not found!')
        for root, dirs, files in os.walk('/kaggle/input'):
            depth = root.replace('/kaggle/input', '').count(os.sep)
            if depth < 4:
                print(f'  {root}/ ({len(files)} files, dirs={dirs[:5]})')
else:
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
    sys.path.insert(0, REPO_ROOT)
    print(f'Solver path: {REPO_ROOT}')
    solver_found = True

# Verify import
try:
    import loopagi.arc.hf_bridge
    print('Import check: loopagi.arc.hf_bridge OK')
except ImportError as e:
    print(f'Import check FAILED: {e}')
    for p in sys.path[:5]:
        if os.path.exists(p):
            print(f'  sys.path entry: {p} -> {os.listdir(p)[:10]}')

In [ ]:
# Cell 2: GPU detection and bitsandbytes install (conditional)
import subprocess, shutil
import torch

# Detect device mode: cuda_bnb, cuda_fp16, or cpu
DEVICE_MODE = 'cpu'
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    cc = major * 10 + minor
    print(f'GPU: {gpu_name}, compute capability: {major}.{minor} (sm_{cc})')
    
    # Test if CUDA actually works (PyTorch 2.10+cu128 dropped sm_60/P100)
    try:
        x = torch.randn(2, 2, device='cuda:0')
        _ = x @ x.T
        del x
        torch.cuda.empty_cache()
        print('CUDA sanity test: PASSED')
        if cc >= 70:
            DEVICE_MODE = 'cuda_bnb'
            print(f'Device mode: cuda_bnb (BnB 4-bit on sm_{cc})')
        else:
            DEVICE_MODE = 'cuda_fp16'
            print(f'Device mode: cuda_fp16 (float16 on sm_{cc})')
    except RuntimeError as e:
        print(f'CUDA sanity test: FAILED - {str(e)[:120]}')
        print(f'*** PyTorch {torch.__version__} cannot execute CUDA ops on {gpu_name} ***')
        print('Device mode: cpu (CUDA broken, falling back to CPU inference)')
        DEVICE_MODE = 'cpu'
else:
    print('No CUDA GPU available')
    print('Device mode: cpu')

# Only install bitsandbytes if we can actually use it
if ON_KAGGLE and DEVICE_MODE == 'cuda_bnb':
    try:
        import bitsandbytes
        print(f'bitsandbytes already installed: {bitsandbytes.__version__}')
    except (ImportError, RuntimeError):
        print('Installing bitsandbytes from offline wheel...')
        BNB_WHEEL = None
        for root, _, files in os.walk('/kaggle/input'):
            for f in files:
                if f.startswith('bitsandbytes') and f.endswith('.whl'):
                    BNB_WHEEL = os.path.join(root, f)
                    break
            if BNB_WHEEL:
                break
        
        if BNB_WHEEL:
            print(f'Found wheel: {BNB_WHEEL}')
            cmd = [sys.executable, '-m', 'pip', 'install', '--no-deps', BNB_WHEEL]
            result = subprocess.run(cmd, capture_output=True, text=True)
            print(result.stdout[-300:] if result.stdout else '')
            if result.returncode != 0:
                print(f'Install error: {result.stderr[-300:]}')
                DEVICE_MODE = 'cpu'
        else:
            print('No bitsandbytes wheel found, downgrading to CPU mode')
            DEVICE_MODE = 'cpu'
elif DEVICE_MODE == 'cpu':
    print('Skipping bitsandbytes (CPU mode - no GPU acceleration)')
    if ON_KAGGLE:
        import psutil
        ram_gb = psutil.virtual_memory().total / 1e9
        print(f'System RAM: {ram_gb:.1f} GB (need ~16GB for Qwen3-8B float32)')
else:
    try:
        import bitsandbytes
        print(f'bitsandbytes: {bitsandbytes.__version__}')
    except ImportError:
        print('bitsandbytes not installed locally')

print(f'\nFinal device mode: {DEVICE_MODE}')

In [ ]:
# Cell 3: Device summary
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    n_gpus = torch.cuda.device_count()
    print(f'GPU: {gpu_name} (x{n_gpus}), VRAM: {vram / 1e9:.1f} GB per GPU')

print(f'Device mode: {DEVICE_MODE}')
if DEVICE_MODE == 'cpu':
    print('*** CPU INFERENCE MODE ***')
    print('*** Model will load in float32 on CPU. Inference will be slow (~5min/call). ***')
    print('*** Only strict transduction phase will be used to stay within 12h limit. ***')

In [ ]:
# Cell 3: Load model
MODEL_NAME = 'unsloth/Qwen3-8B-unsloth-bnb-4bit'

# Check if model is available as Kaggle dataset (pre-downloaded, no internet needed)
if ON_KAGGLE:
    # Search for config.json (model marker) under /kaggle/input
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'config.json' in files and 'model.safetensors' in files:
            MODEL_NAME = root
            print(f'Found model at: {root}')
            break
        elif 'config.json' in files and any(f.endswith('.safetensors') for f in files):
            MODEL_NAME = root
            print(f'Found model at: {root}')
            break
    else:
        # Fallback: try known paths
        for model_path in [
            '/kaggle/input/qwen3-8b-unsloth-4bit-quantized',
        ]:
            if os.path.exists(model_path):
                MODEL_NAME = model_path
                print(f'Using model dir: {model_path}')
                print(f'Contents: {os.listdir(model_path)[:10]}')
                break

print(f'Loading model: {MODEL_NAME}')
model_start = time.monotonic()

from loopagi.arc.hf_bridge import create_hf_bridge
bridge = create_hf_bridge(model=MODEL_NAME)

model_time = time.monotonic() - model_start
print(f'Model loaded in {model_time:.1f}s')

# Quick test
test_response = bridge.call('What is 2+2? Answer with just the number.', temperature=0.0)
print(f'Test response: {test_response[:50]}')

In [ ]:
# Cell 4: Load competition data
if ON_KAGGLE:
    # Kaggle mounts competition data under /kaggle/input/competitions/
    DATA_DIR = '/kaggle/input/competitions/arc-prize-2026-arc-agi-2'
    if not os.path.exists(DATA_DIR):
        # Fallback: try direct path
        DATA_DIR = '/kaggle/input/arc-prize-2026-arc-agi-2'
    if not os.path.exists(DATA_DIR):
        # Search for the challenge file
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'arc-agi_test_challenges.json' in files:
                DATA_DIR = root
                break
else:
    DATA_DIR = '/tmp/arc-prize-2026-data'

print(f'Data dir: {DATA_DIR}')
print(f'Contents: {os.listdir(DATA_DIR)}')

# Competition uses test_challenges for actual submission
CHALLENGES_PATH = os.path.join(DATA_DIR, 'arc-agi_test_challenges.json')
if not os.path.exists(CHALLENGES_PATH):
    # Fallback to evaluation challenges (for testing)
    CHALLENGES_PATH = os.path.join(DATA_DIR, 'arc-agi_evaluation_challenges.json')
    print(f'WARNING: test_challenges not found, using evaluation_challenges')

SAMPLE_SUB_PATH = os.path.join(DATA_DIR, 'sample_submission.json')

with open(CHALLENGES_PATH) as f:
    tasks = json.load(f)

# Load sample submission if available, otherwise create empty
if os.path.exists(SAMPLE_SUB_PATH):
    with open(SAMPLE_SUB_PATH) as f:
        sample_sub = json.load(f)
else:
    sample_sub = {tid: None for tid in tasks}

print(f'Loaded {len(tasks)} tasks from {os.path.basename(CHALLENGES_PATH)}')

In [ ]:
# Cell 5: Configure solver - v13 SPEED MODE
# v13 fixes:
#   1. Disabled augmented transduction (D4 voting = 16 extra LLM calls per failed task)
#   2. Disabled refined transduction (iterative refinement burns time on failures)
#   3. Disabled relaxed retry at 0.90 (only try 0.97 threshold)
#   4. Reduced per-task timeout from 180s to 90s
#   5. adaptive=False prevents overriding max_hyp/iter
from loopagi.arc.solve_improved import solve_task_improved, ImprovedSolverConfig
from loopagi.arc.arc_loader import ArcTask, GridPair, TestInput

config = ImprovedSolverConfig(
    enable_transduction=True,
    enable_augmented_transduction=False,   # v13: skip D4 voting (saves 10-20 min per failed task)
    enable_refined_transduction=False,     # v13: skip iterative refinement
    enable_relaxed_retry=False,            # v13: don't retry relaxed at 0.90
    enable_evolution=False,
    enable_sampling=False,
    enable_ttt=False,
    enable_multi_strategy=False,
    enable_diff_refine=False,
    enable_nl_evolution=False,
    enable_nl_description=False,
    enable_object_prompts=False,
    relaxed_auto_accept=0.95,
    sample_n=1,
    cell_fix_threshold=0.90,
)

MAX_HYPOTHESES = 1
MAX_ITERATIONS = 1
MAX_TOTAL_SECONDS = 40000  # ~11.1h
MAX_PER_TASK_SECONDS = 90  # v13: reduced from 180s to 90s

print('Solver configured (v13 SPEED MODE):')
print(f'  Transduction: {config.enable_transduction}')
print(f'  Augmented transduction (D4): {config.enable_augmented_transduction}')
print(f'  Refined transduction: {config.enable_refined_transduction}')
print(f'  Relaxed retry (0.90): {config.enable_relaxed_retry}')
print(f'  Multi-strategy: {config.enable_multi_strategy}')
print(f'  Sampling: {config.enable_sampling}')
print(f'  NL description: {config.enable_nl_description}')
print(f'  Evolution: {config.enable_evolution}')
print(f'  TTT: {config.enable_ttt}')
print(f'  Max hypotheses: {MAX_HYPOTHESES}')
print(f'  Max iterations: {MAX_ITERATIONS}')
print(f'  Per-task timeout: {MAX_PER_TASK_SECONDS}s')
print(f'  Adaptive: DISABLED')

In [ ]:
# Cell 6: Helper functions
def make_task(task_id, task_data):
    """Convert Kaggle JSON task to our ArcTask dataclass."""
    return ArcTask(
        task_id=task_id,
        train=[GridPair(input=p['input'], output=p['output']) for p in task_data['train']],
        test=[TestInput(input=t['input'], output=t.get('output')) for t in task_data['test']],
    )

def build_submission_entry(task, result):
    """Convert solver result to Kaggle submission format (2 attempts per test)."""
    entries = []
    predictions = getattr(result, 'predictions', [])
    for idx, test in enumerate(task.test):
        if idx < len(predictions):
            pred = predictions[idx]
            if isinstance(pred, list) and pred and isinstance(pred[0], list):
                if isinstance(pred[0][0], list):
                    entries.append({
                        'attempt_1': pred[0],
                        'attempt_2': pred[1] if len(pred) > 1 else pred[0],
                    })
                else:
                    entries.append({'attempt_1': pred, 'attempt_2': pred})
            else:
                entries.append({'attempt_1': pred, 'attempt_2': pred})
        else:
            entries.append({'attempt_1': test.input, 'attempt_2': test.input})
    return entries

def fallback_entry(task_data):
    """Identity fallback: output = input."""
    return [{'attempt_1': t['input'], 'attempt_2': t['input']} for t in task_data['test']]

print('Helper functions defined.')

In [ ]:
# Cell 7: Run solver on all tasks - v13 with OOM prevention
# v13 fixes:
#   1. gc.collect() + torch.cuda.empty_cache() between every task
#   2. Per-task timeout reduced to 90s
#   3. Skip tasks immediately if total budget nearly exhausted
#   4. Log VRAM usage between tasks for debugging
import gc
import traceback

submission = {}
total_start = time.monotonic()
solved_count = 0
error_count = 0
oom_count = 0
n_tasks = len(tasks)

print(f'Starting: {n_tasks} tasks (max {MAX_PER_TASK_SECONDS}s per task)')
print(f'adaptive=False, max_hyp={MAX_HYPOTHESES}, max_iter={MAX_ITERATIONS}')
print(f'v13: augmented_transduction=OFF, refined_transduction=OFF, relaxed_retry=OFF')
print('=' * 60)

for i, (task_id, task_data) in enumerate(tasks.items()):
    elapsed_total = time.monotonic() - total_start
    remaining = MAX_TOTAL_SECONDS - elapsed_total

    # Time budget check
    if remaining < 120:
        print(f'\nTime budget exhausted at task {i+1}/{n_tasks}, filling rest with fallback')
        for tid, tdata in list(tasks.items())[i:]:
            submission[tid] = fallback_entry(tdata)
        break

    task = make_task(task_id, task_data)
    task_start = time.monotonic()

    try:
        solve_dict = solve_task_improved(
            task=task,
            bridge=bridge,
            max_hypotheses=MAX_HYPOTHESES,
            max_iterations=MAX_ITERATIONS,
            adaptive=False,
            config=config,
            analytics=None,
        )
        result = solve_dict['result']
        task_time = time.monotonic() - task_start

        if result.solved:
            solved_count += 1
            status = f'SOLVED [{solve_dict.get("complexity", "?")}]'
        else:
            status = f'{result.best_similarity:.0%}'

        print(f'[{i+1:3d}/{n_tasks}] {task_id}: {status} ({task_time:.1f}s) | total: {elapsed_total/60:.0f}m | solved: {solved_count}')
        submission[task_id] = build_submission_entry(task, result)

    except Exception as e:
        task_time = time.monotonic() - task_start
        error_count += 1
        err_str = str(e)
        if 'CUDA out of memory' in err_str:
            oom_count += 1
        print(f'[{i+1:3d}/{n_tasks}] {task_id}: ERROR {err_str[:80]} ({task_time:.1f}s)')
        submission[task_id] = fallback_entry(task_data)

    # v13: Aggressive memory cleanup between tasks
    gc.collect()
    try:
        torch.cuda.empty_cache()
        # Log VRAM state every 10 tasks
        if (i + 1) % 10 == 0 and torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated(0) / 1e9
            reserved = torch.cuda.memory_reserved(0) / 1e9
            print(f'  [VRAM] allocated={allocated:.2f}GB, reserved={reserved:.2f}GB')
    except Exception:
        pass

total_time = time.monotonic() - total_start
print('\n' + '=' * 60)
print(f'Done: {solved_count}/{n_tasks} solved ({solved_count/n_tasks:.1%})')
print(f'Errors: {error_count} (OOM: {oom_count})')
print(f'Total time: {total_time/3600:.1f}h ({total_time:.0f}s)')
print(f'Avg per task: {total_time/max(i+1,1):.1f}s')
print(f'LLM stats: {bridge.stats()}')

In [ ]:
# Cell 8: Validate and save submission
# Ensure all tasks are present
missing = set(sample_sub.keys()) - set(submission.keys())
if missing:
    print(f'WARNING: {len(missing)} tasks missing, adding fallback')
    for tid in missing:
        submission[tid] = fallback_entry(tasks[tid])

# Validate format
errors = []
for tid, entries in submission.items():
    if not isinstance(entries, list):
        errors.append(f'{tid}: entries is not a list')
        continue
    for j, entry in enumerate(entries):
        if 'attempt_1' not in entry or 'attempt_2' not in entry:
            errors.append(f'{tid}[{j}]: missing attempt_1 or attempt_2')
        for key in ['attempt_1', 'attempt_2']:
            grid = entry.get(key)
            if not isinstance(grid, list) or not grid or not isinstance(grid[0], list):
                errors.append(f'{tid}[{j}].{key}: not a valid 2D grid')

if errors:
    print(f'Validation errors ({len(errors)}):')
    for e in errors[:10]:
        print(f'  {e}')
else:
    print(f'Validation passed: {len(submission)} tasks, all have attempt_1 + attempt_2')

# Save
OUTPUT_PATH = '/kaggle/working/submission.json' if ON_KAGGLE else 'submission.json'
with open(OUTPUT_PATH, 'w') as f:
    json.dump(submission, f)

file_size = os.path.getsize(OUTPUT_PATH)
print(f'Saved to {OUTPUT_PATH} ({file_size/1024:.1f} KB)')